# Elastic Net Regression Code Companion

Elastic Net combines Lasso-style feature selection and Ridge-style coefficient stability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_regression(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    }


## Load and Split Data


In [ ]:
data = load_diabetes(as_frame=True)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)


## Train Ridge, Lasso, and Elastic Net


In [ ]:
models = {
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=1.0, max_iter=10000))]),
    "Elastic Net": Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000))])
}
for model in models.values():
    model.fit(X_train, y_train)


In [ ]:
pd.DataFrame([evaluate_regression(name, model, X_test, y_test) for name, model in models.items()])


## Inspect Coefficients

Elastic Net can shrink coefficients and may set some to zero depending on `alpha` and `l1_ratio`.


In [ ]:
coef = models["Elastic Net"].named_steps["model"].coef_
pd.DataFrame({"feature": X.columns, "elastic_net_coefficient": coef, "selected": coef != 0})


## Effect of L1 Ratio


In [ ]:
rows=[]
for ratio in [0.1, 0.3, 0.5, 0.7, 0.9]:
    model=Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(alpha=0.5, l1_ratio=ratio, max_iter=10000))])
    model.fit(X_train,y_train)
    row=evaluate_regression(f"l1_ratio={ratio}", model, X_test, y_test)
    row["selected_features"] = int((model.named_steps["model"].coef_ != 0).sum())
    rows.append(row)
pd.DataFrame(rows)
